<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/01e-introduction-to-pytorch-lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to (PyTorch) Lightning

The material covered in this notebook is derived from the [*Lightning in 15 minutes*](https://lightning.ai/docs/pytorch/stable/starter/introduction.html) tutorial.

## Step 0: Install Lightning

In [ ]:
%%bash

pip install lightning torchinfo

In [ ]:
import pathlib


import lightning as L
import torch
from torch import nn, optim, utils
import torchvision
import torchvision.transforms.v2 as T


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
print(DEVICE)

## Step 1: Creating a `LightningModule`

### Wrap your PyTorch model inside a `LightningModule`

In [ ]:
class LightningClassifier(L.LightningModule):

    def __init__(self, lr=1e-3):
        super().__init__()

        # store hyperparameters provided to constructor!
        self.lr = lr

        # use save_hyperparameters to store the learning rate and other args!
        self.save_hyperparameters()

        # neural network architecture defined in the __init__ method!
        self.model_fn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 100),
            nn.ReLU(),
            nn.Linear(100, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
        self.model_fn.to(DEVICE)

        # loss function is also defined in the __init__method!
        self.loss_fn = nn.CrossEntropyLoss()

### Define prediction/inference logic using the `forward` hook

In [ ]:
class LightningClassifier(L.LightningModule):

    def __init__(self, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.save_hyperparameters()
        self.model_fn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 100),
            nn.ReLU(),
            nn.Linear(100, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
        self.model_fn.to(DEVICE)
        self.loss_fn = nn.CrossEntropyLoss()

    # the forward method defines neural network prediction/inference
    def forward(self, X):
        return self.model_fn(X)


### Configure optimizers using the `configure_optimizer` hook

In [ ]:
class LightningClassifier(L.LightningModule):

    def __init__(self, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.save_hyperparameters()
        self.model_fn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 100),
            nn.ReLU(),
            nn.Linear(100, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
        self.model_fn.to(DEVICE)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, X):
        return self.model_fn(X)

    # self.parameters will contain all parameters for all modules defined in __init__
    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(),
            lr=self.lr,
        )
        return optimizer


### Training logic goes into the `training_step` hook

In [ ]:
class LightningClassifier(L.LightningModule):

    def __init__(self, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.save_hyperparameters()
        self.model_fn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 100),
            nn.ReLU(),
            nn.Linear(100, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
        self.model_fn.to(DEVICE)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, X):
        return self.model_fn(X)

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(),
            lr=self.lr,
        )
        return optimizer

    def training_step(self, train_batch, batch_idx):
        X, y = train_batch
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)

        # log information to Tensorboard (or preferred logger)
        self.log(
            "train_loss",
            loss,
            on_epoch=True,  # calculate epoch level-metrics
            prog_bar=True
        )
        return loss

### Validation logic goes into the `validation_step` hook

In [ ]:
class LightningClassifier(L.LightningModule):

    def __init__(self, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.save_hyperparameters()
        self.model_fn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 100),
            nn.ReLU(),
            nn.Linear(100, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
        self.model_fn.to(DEVICE)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, X):
        return self.model_fn(X)

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(),
            lr=self.lr,
        )
        return optimizer

    def training_step(self, train_batch, batch_idx):
        X, y = train_batch
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "train_loss",
            loss,
            on_epoch=True,
            prog_bar=True
        )
        return loss

    def validation_step(self, val_batch, batch_idx):
        X, y = val_batch
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)

        # validation_step automatically computes epoch-level metrics
        self.log(
            "val_loss",
            loss,
            prog_bar=True
        )


### Test logic goes into the `test_step` hook

In [ ]:
class LightningClassifier(L.LightningModule):

    def __init__(self, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.save_hyperparameters()
        self.model_fn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 100),
            nn.ReLU(),
            nn.Linear(100, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
        self.model_fn.to(DEVICE)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, X):
        return self.model_fn(X)

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(),
            lr=self.lr,
        )
        return optimizer

    def training_step(self, train_batch, batch_idx):
        X, y = train_batch
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "train_loss",
            loss,
            on_epoch=True,
            prog_bar=True
        )
        return loss

    def validation_step(self, val_batch, batch_idx):
        X, y = val_batch
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "val_loss",
            loss,
            prog_bar=True
        )

    def test_step(self, test_batch, batch_idx):
        X, y = test_batch
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "test_loss",
            loss,
            prog_bar=True
        )


### Predict logic goes into the `predict_step` hook

In [ ]:
class LightningClassifier(L.LightningModule):

    def __init__(self, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.save_hyperparameters()
        self.model_fn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 100),
            nn.ReLU(),
            nn.Linear(100, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
        self.model_fn.to(DEVICE)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, X):
        return self.model_fn(X)

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(),
            lr=self.lr,
        )
        return optimizer

    def training_step(self, train_batch, batch_idx):
        X, y = train_batch
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "train_loss",
            loss,
            on_epoch=True,
            prog_bar=True
        )
        return loss

    def validation_step(self, val_batch, batch_idx):
        X, y = val_batch
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "val_loss",
            loss,
            prog_bar=True
        )

    def test_step(self, test_batch, batch_idx):
        X, y = test_batch
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "test_loss",
            loss,
            prog_bar=True
        )

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        X, *_ = batch
        logits = self.forward(X)
        y_probas = torch.softmax(logits, dim=1)
        return y_probas




### Remove any `cuda()` or `to(DEVICE)` calls

In [ ]:
class LightningClassifier(L.LightningModule):

    def __init__(self, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.save_hyperparameters()
        self.model_fn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 100),
            nn.ReLU(),
            nn.Linear(100, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, X):
        return self.model_fn(X)

    def configure_optimizers(self):
        return optim.AdamW(
            self.parameters(),
            lr=self.lr,
        )

    def training_step(self, train_batch, batch_idx):
        X, y = train_batch
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "train_loss",
            loss,
            on_epoch=True,
            prog_bar=True
        )
        return loss

    def validation_step(self, val_batch, batch_idx):
        X, y = val_batch
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "val_loss",
            loss,
            prog_bar=True
        )

    def test_step(self, test_batch, batch_idx):
        X, y = test_batch
        logits = self.forward(X)
        loss = self.loss_fn(logits, y)
        self.log(
            "test_loss",
            loss,
            prog_bar=True
        )

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        X, *_ = batch
        logits = self.forward(X)
        y_probas = torch.softmax(logits, dim=1)
        return y_probas

## Step 2: Define a `LightningDataModule`

### Preparing data

In [ ]:
class TorchvisionDataModule(L.LightningDataModule):

    def __init__(self, data_dir: pathlib.Path):
        super().__init__()

        self.data_dir = data_dir

    def prepare_data(self) -> None:
        """Called once per node to download data."""
        torchvision.datasets.MNIST(
            self.data_dir,
            download=True,
            train=True,
        )

        torchvision.datasets.MNIST(
            self.data_dir,
            download=True,
            train=False,
        )

### Setup the DataSets

In [ ]:
from typing import Optional


class TorchvisionDataModule(L.LightningDataModule):

    def __init__(self, data_dir: pathlib.Path, seed: int, val_size: float):
        super().__init__()

        self.data_dir = data_dir
        self.seed = seed
        self.transform = T.Compose(
            [
                T.ToImage(),
                T.ToDtype(torch.float32, scale=True),
            ]
        )
        self.val_size = val_size

    def prepare_data(self) -> None:
        """Called once per node to download data."""
        torchvision.datasets.MNIST(
            self.data_dir,
            download=True,
            train=True,
        )

        torchvision.datasets.MNIST(
            self.data_dir,
            download=True,
            train=False,
        )

    def setup(self, stage: Optional[str]=None) -> None:
        """Called once per GPU to setup data."""

        if stage == "fit" or stage is None:
            train_val_dataset = (
                torchvision.datasets
                           .MNIST(
                               self.data_dir,
                               train=True,
                               download=False,
                               transform=self.transform
                            )
            )

            # train/val split
            generator = torch.Generator().manual_seed(self.seed)
            n_samples = len(train_val_dataset)
            n_val_samples = int(self.val_size * n_samples)
            n_train_samples = n_samples - n_val_samples
            self.train_dataset, self.val_dataset = (
                utils.data
                    .random_split(
                        train_val_dataset,
                        [n_train_samples, n_val_samples],
                        generator
                    )
            )

        if stage == "test" or stage is None:
            self.test_dataset = (
                torchvision.datasets
                           .MNIST(
                               self.data_dir,
                               train=False,
                               download=False,
                               transform=self.transform
                           )
            )





### Configure the DataLoaders

In [ ]:
from typing import Optional


class TorchvisionDataModule(L.LightningDataModule):

    def __init__(self, data_dir: pathlib.Path, seed: int, val_size: float):
        super().__init__()

        self.data_dir = data_dir
        self.data_loader_kwargs = {
            "batch_size": 32,
            "num_workers": 2,            # load data in parallel using multiple workers
            "persistent_workers": True,  # keep workers around between epochs
            "pin_memory": False,         # avoid extra copy of data batches when using GPU
            "prefetch_factor": 2,        # fetch multiple data batches in advance
        }
        self.seed = seed
        self.transform = T.Compose(
            [
                T.ToImage(),
                T.ToDtype(torch.float32, scale=True),
            ]
        )
        self.val_size = val_size

    def prepare_data(self) -> None:
        """Called once per node to download data."""
        torchvision.datasets.MNIST(
            self.data_dir,
            download=True,
            train=True,
        )

        torchvision.datasets.MNIST(
            self.data_dir,
            download=True,
            train=False,
        )

    def setup(self, stage: Optional[str]=None) -> None:
        """Called once per GPU to setup data."""

        if stage == "fit" or stage is None:
            train_val_dataset = (
                torchvision.datasets
                           .MNIST(
                               self.data_dir,
                               train=True,
                               download=False,
                               transform=self.transform
                           )
            )

            # train/val split
            generator = torch.Generator().manual_seed(self.seed)
            n_samples = len(train_val_dataset)
            n_val_samples = int(self.val_size * n_samples)
            n_train_samples = n_samples - n_val_samples
            self.train_dataset, self.val_dataset = (
                utils.data
                    .random_split(
                        train_val_dataset,
                        [n_train_samples, n_val_samples],
                        generator
                    )
            )

        if stage == "test" or stage is None:
            self.test_dataset = (
                torchvision.datasets
                           .MNIST(
                               self.data_dir,
                               train=False,
                               download=False,
                               transform=self.transform
                           )
            )

    def train_dataloader(self) -> utils.data.DataLoader:
        return utils.data.DataLoader(
            self.train_dataset,
            shuffle=True,
            **self.data_loader_kwargs
        )

    def val_dataloader(self) -> utils.data.DataLoader:
        return utils.data.DataLoader(
            self.val_dataset,
            shuffle=False,
            **self.data_loader_kwargs
        )

    def test_dataloader(self) -> utils.data.DataLoader:
        return utils.data.DataLoader(
            self.test_dataset,
            shuffle=False,
            **self.data_loader_kwargs
        )




## Step 3: Use the `Trainer` to train and evaluate your `LightningModule`

### Initialize your `LightningModule`

In [ ]:
model_fn = LightningClassifier()

### Initialize your `LightningDataModule`

In [ ]:
mnist_datamodule = TorchvisionDataModule(
    data_dir=pathlib.Path("./sample_data"),
    seed=42,
    val_size=0.2,
)

### Initialize a `Trainer`

The Lightning Trainer automates all the engineering including:

* training and evaluation loops
* hardware calls
* switching between train and eval modes
* zeroing out the gradient

In [ ]:
trainer = L.Trainer(
    max_epochs=10,
)

In [ ]:
trainer.fit(
    model_fn,
    datamodule=mnist_datamodule,
)

In [ ]:
trainer.test(
    model_fn,
    datamodule=mnist_datamodule
)